In [3]:
import pandas as pd
import numpy as np
df=pd.read_csv("../data/bank_churn.csv")
df.head()

,id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,0,15674932,Okwudilichukwu,668,France,Male,33.0,3,0.00,2,1.0,0.0,181449.97,0
1,1,15749177,Okwudiliolisa,627,France,Male,33.0,1,0.00,2,1.0,1.0,49503.50,0
2,2,15694510,Hsueh,678,France,Male,40.0,10,0.00,2,1.0,0.0,184866.69,0
3,3,15741417,Kao,581,France,Male,34.0,2,148882.54,1,1.0,1.0,84560.88,0
4,4,15766172,Chiemenam,716,Spain,Male,33.0,5,0.00,2,1.0,1.0,15068.83,0


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 165034 entries, 0 to 165033
Data columns (total 14 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   id               165034 non-null  int64  
 1   CustomerId       165034 non-null  int64  
 2   Surname          165034 non-null  str    
 3   CreditScore      165034 non-null  int64  
 4   Geography        165034 non-null  str    
 5   Gender           165034 non-null  str    
 6   Age              165034 non-null  float64
 7   Tenure           165034 non-null  int64  
 8   Balance          165034 non-null  float64
 9   NumOfProducts    165034 non-null  int64  
 10  HasCrCard        165034 non-null  float64
 11  IsActiveMember   165034 non-null  float64
 12  EstimatedSalary  165034 non-null  float64
 13  Exited           165034 non-null  int64  
dtypes: float64(5), int64(6), str(3)
memory usage: 20.4 MB


In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
print("Customer Churn Distribution:")
print(df["Exited"].value_counts())

print("\nChurn Percentage:")
print((df["Exited"].value_counts(normalize=True) * 100).round(2))

Customer Churn Distribution:
Exited
0    130113
1     34921
Name: count, dtype: int64

Churn Percentage:
Exited
0    78.84
1    21.16
Name: proportion, dtype: float64


In [7]:
df=df.drop(columns=["id","CustomerId","Surname"])
X=df.drop(columns=["Exited"])
y=df["Exited"]

In [8]:
from sklearn.model_selection import train_test_split

# First split: 70% training, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Second split: 15% validation, 15% testing
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

Training: (115523, 10)
Validation: (24755, 10)
Testing: (24756, 10)


In [9]:
numerical_features = [
    "CreditScore", "Age", "Tenure",
    "Balance", "NumOfProducts",
    "HasCrCard", "IsActiveMember", "EstimatedSalary"
]

categorical_features = ["Geography", "Gender"]

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Numerical preprocessing
numerical_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both transformers
preprocessor = ColumnTransformer(transformers=[
    ("num", numerical_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

In [11]:
X_train_processed=preprocessor.fit_transform(X_train)
X_val_processed=preprocessor.transform(X_val)
X_test_processed=preprocessor.transform(X_test)
print("Training shape:", X_train_processed.shape)
print("Validation shape:", X_val_processed.shape)
print("Testing shape:", X_test_processed.shape)

Training shape: (115523, 13)
Validation shape: (24755, 13)
Testing shape: (24756, 13)


In [12]:
import joblib
from pathlib import Path

# Ensure models directory exists
Path("../models").mkdir(parents=True, exist_ok=True)

# Save fitted preprocessing pipeline
joblib.dump(preprocessor, "../models/preprocessor.joblib")

print("Preprocessing pipeline saved successfully!")

Preprocessing pipeline saved successfully!
